In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [10]:
import joblib
from src.config import MODEL_DIR
from src.evaluate import get_oof_predictions

print(sorted(path.name for path in MODEL_DIR.glob("*.joblib")))

['decision_tree_baseline.joblib', 'lightgbm_0.025learn.joblib', 'lightgbm_127leaves.joblib', 'lightgbm_63leaves.joblib', 'lightgbm_encoded.joblib', 'lightgbm_encoded_no.joblib', 'lightgbm_numeric.joblib', 'lightgbm_numeric_no.joblib', 'lightgbm_oof.joblib', 'logreg_baseline.joblib', 'logreg_onehot_ratings.joblib', 'random_forest.joblib', 'random_forest_without_onehot.joblib', 'xgboost.joblib']


### Choosing the candidates

In [3]:
candidates = ["lightgbm_0.025learn", "xgboost", "random_forest_without_onehot", "logreg_onehot_ratings"]

for name in candidates:
    saved = joblib.load(MODEL_DIR / f"{name}.joblib")
    print(name, "->", type(saved).__name__, list(saved.keys()) if isinstance(saved, dict) else "")
    if isinstance(saved, dict) and "state" in saved:
        print("    ratings mode:", saved["state"]["ratings"])

lightgbm_0.025learn -> dict ['state', 'model']
    ratings mode: numeric
xgboost -> dict ['state', 'model']
    ratings mode: numeric
random_forest_without_onehot -> dict ['state', 'model']
    ratings mode: numeric
logreg_onehot_ratings -> dict ['state', 'model']
    ratings mode: onehot


### Imports

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src.data import load_train, get_split
from src.preprocessing import apply_preprocessing

In [5]:
X, y = load_train()
X_tr, X_val, y_tr, y_val = get_split(X, y)

val_probas = {}

for name in candidates:
    saved = joblib.load(MODEL_DIR / f"{name}.joblib")
    X_val_encoded = apply_preprocessing(X_val, saved["state"])
    val_probas[name] = saved["model"].predict_proba(X_val_encoded)[:, 1]
    print(f"{name:32s} val AUC = {roc_auc_score(y_val, val_probas[name]):.4f}")

lightgbm_0.025learn              val AUC = 0.9576
xgboost                          val AUC = 0.9572
random_forest_without_onehot     val AUC = 0.9544
logreg_onehot_ratings            val AUC = 0.9476


### Blend


In [6]:
def auc(p):
    return round(roc_auc_score(y_val, p), 5)

lgbm = val_probas["lightgbm_0.025learn"]
xgb = val_probas["xgboost"]
rf = val_probas["random_forest_without_onehot"]
lr = val_probas["logreg_onehot_ratings"]

print("LightGBM alone       :", auc(lgbm))
print("LGBM + XGB           :", auc((lgbm + xgb) / 2))
print("LGBM + RF            :", auc((lgbm + rf) / 2))
print("LGBM + XGB + RF      :", auc((lgbm + xgb + rf) / 3))
print("LGBM + XGB + RF + LR :", auc((lgbm + xgb + rf + lr) / 4))

LightGBM alone       : 0.9576
LGBM + XGB           : 0.95767
LGBM + RF            : 0.95646
LGBM + XGB + RF      : 0.95697
LGBM + XGB + RF + LR : 0.9558


### Submission

In [7]:
from src.data import load_test
from src.submission import save_submission

test_ids, X_test = load_test()

test_probas = {}
for name in ["lightgbm_0.025learn", "xgboost"]:
    saved = joblib.load(MODEL_DIR / f"{name}.joblib")
    X_test_encoded = apply_preprocessing(X_test, saved["state"])
    test_probas[name] = saved["model"].predict_proba(X_test_encoded)[:, 1]
    print(name, X_test_encoded.shape)

lightgbm_0.025learn (299844, 39)
xgboost (299844, 39)


In [8]:
blend = (test_probas["lightgbm_0.025learn"] + test_probas["xgboost"]) / 2

print("correlation between the two:", np.corrcoef(test_probas["lightgbm_0.025learn"], test_probas["xgboost"])[0, 1].round(4))

save_submission(test_ids, blend, "lgbm_xgb_blend.csv")

correlation between the two: 0.998
Saved to: C:\Users\Bharat\Desktop\remaining-2026\Career-Tracks\Track 01 - Data Science\Relevant Projects\airline-satisfaction-prediction\submissions\lgbm_xgb_blend.csv
count    299844.000000
mean          0.443560
std           0.429507
min           0.001820
25%           0.036468
50%           0.192036
75%           0.948358
max           0.992782
Name: satisfaction, dtype: float64


,id,satisfaction
0,699635,0.238557
1,699636,0.851573
2,699637,0.904180
3,699638,0.061355
4,699639,0.050520
...,...,...
299839,999474,0.042786
299840,999475,0.041143
299841,999476,0.299489
299842,999477,0.028650


In [9]:
saved = joblib.load(MODEL_DIR / "lightgbm_oof.joblib")
print(type(saved))
print(saved.keys() if isinstance(saved, dict) else getattr(saved, "shape", None))

<class 'dict'>
dict_keys(['state', 'model'])


In [12]:
from src.models import make_lightgbm
X, y = load_train()
test_ids, X_test = load_test()



oof, test_pred = get_oof_predictions(make_lightgbm, X, y, X_test)

np.save(MODEL_DIR / "oof_lightgbm.npy", oof)
np.save(MODEL_DIR / "test_lightgbm.npy", test_pred)

fold 1: AUC = 0.95831
fold 2: AUC = 0.95713
fold 3: AUC = 0.95905
fold 4: AUC = 0.95822
fold 5: AUC = 0.95850
pooled OOF AUC = 0.95823


In [13]:
from src.submission import save_submission

save_submission(test_ids, test_pred, "lightgbm_5fold.csv")

Saved to: C:\Users\Bharat\Desktop\remaining-2026\Career-Tracks\Track 01 - Data Science\Relevant Projects\airline-satisfaction-prediction\submissions\lightgbm_5fold.csv
count    299844.000000
mean          0.443748
std           0.428876
min           0.003990
25%           0.036966
50%           0.198099
75%           0.949097
max           0.991305
Name: satisfaction, dtype: float64


,id,satisfaction
0,699635,0.172860
1,699636,0.910185
2,699637,0.900154
3,699638,0.077923
4,699639,0.045961
...,...,...
299839,999474,0.052002
299840,999475,0.033159
299841,999476,0.276833
299842,999477,0.028288
